Install and Imports

In [20]:
!pip install scikit-learn joblib scipy

import joblib
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from scipy.sparse import issparse


Load Feature Matrix/Data

In [21]:
# Load feature matrix (X), labels (y), and preprocessing objects
X, y, tfidf, scaler = joblib.load("feature_matrix.pkl")

print("Type of X:", type(X))
if hasattr(X, "shape"):
    print("Shape of X:", X.shape)

print("Shape of y:", np.shape(y))

# If you used 0=Low, 1=Medium, 2=High
label_map_inv = {0: "Low", 1: "Medium", 2: "High"}

unique, counts = np.unique(y, return_counts=True)
print("\nLabel distribution:")
for lbl, cnt in zip(unique, counts):
    print(f"  {lbl} ({label_map_inv.get(int(lbl), '?')}): {cnt}")


Type of X: <class 'scipy.sparse._csr.csr_matrix'>
Shape of X: (5, 232)
Shape of y: (5,)

Label distribution:
  1 (Medium): 1
  2 (High): 4


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code 

Set Up K-Fold Cross Validation

In [22]:
# Use 3-fold CV due to very small sample size
k = 3
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

print(f"Using K-Fold cross-validation with k={k}")


Using K-Fold cross-validation with k=3


Multinomial Logistic Regression (Baseline)

In [23]:
log_reg = LogisticRegression(
    solver="lbfgs",
    max_iter=2000,
    n_jobs=-1
)

scores_lr = cross_val_score(
    log_reg,
    X,
    y,
    cv=kfold,
    scoring='accuracy'
)

print("=== Multinomial Logistic Regression (No PCA) ===")
print("Fold accuracies:", scores_lr)
print("Mean accuracy:", scores_lr.mean())
print("Std deviation:", scores_lr.std())


=== Multinomial Logistic Regression (No PCA) ===
Fold accuracies: [ 1.  1. nan]
Mean accuracy: nan
Std deviation: nan


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
1 fits failed out of a total of 3.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py", line 1301, in fit
    raise ValueError(
ValueEr

Random Forest

In [24]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

scores_rf = cross_val_score(
    rf,
    X,
    y,
    cv=kfold,
    scoring='accuracy'
)

print("=== Random Forest (No PCA) ===")
print("Fold accuracies:", scores_rf)
print("Mean accuracy:", scores_rf.mean())
print("Std deviation:", scores_rf.std())


=== Random Forest (No PCA) ===
Fold accuracies: [1. 1. 0.]
Mean accuracy: 0.6666666666666666
Std deviation: 0.4714045207910317


Gradient Boosting (no PCA, using dense X)

In [25]:
# Convert X to dense only if needed (safe for small datasets)
if issparse(X):
    X_dense = X.toarray()
else:
    X_dense = X

gb = GradientBoostingClassifier(
    random_state=42
)

scores_gb = cross_val_score(
    gb,
    X_dense,
    y,
    cv=kfold,
    scoring='accuracy'
)

print("=== Gradient Boosting (No PCA) ===")
print("Fold accuracies:", scores_gb)
print("Mean accuracy:", scores_gb.mean())
print("Std deviation:", scores_gb.std())


=== Gradient Boosting (No PCA) ===
Fold accuracies: [ 1.  1. nan]
Mean accuracy: nan
Std deviation: nan


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
1 fits failed out of a total of 3.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_gb.py", line 669, in fit
    y = self._encode_y(y=y, sample_weigh

Logistic Regression with SVD (PCA-style)

In [26]:
# Number of components to keep (you can experiment with 3, 5, 10, etc.)
n_components = 5

svd = TruncatedSVD(n_components=n_components, random_state=42)
log_reg_pca = LogisticRegression(
    solver="lbfgs",
    max_iter=2000,
    n_jobs=-1
)

pipeline_lr_pca = Pipeline([
    ("svd", svd),
    ("clf", log_reg_pca)
])

scores_lr_pca = cross_val_score(
    pipeline_lr_pca,
    X,
    y,
    cv=kfold,
    scoring='accuracy'
)

print(f"=== Logistic Regression with SVD (n_components={n_components}) ===")
print("Fold accuracies:", scores_lr_pca)
print("Mean accuracy:", scores_lr_pca.mean())
print("Std deviation:", scores_lr_pca.std())

# Optional: fit once on full data to inspect explained variance
pipeline_lr_pca.fit(X, y)
explained = pipeline_lr_pca.named_steps["svd"].explained_variance_ratio_
print("\nExplained variance by components:", explained)
print("Total explained variance:", explained.sum())


=== Logistic Regression with SVD (n_components=5) ===
Fold accuracies: [ 1.  1. nan]
Mean accuracy: nan
Std deviation: nan

Explained variance by components: [5.15772607e-01 2.34319603e-01 1.53259922e-01 9.66434306e-02
 4.43755651e-06]
Total explained variance: 1.000000000000001


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
1 fits failed out of a total of 3.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last_

Random Forest with SVD

In [27]:
rf_pca = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

pipeline_rf_pca = Pipeline([
    ("svd", TruncatedSVD(n_components=n_components, random_state=42)),
    ("clf", rf_pca)
])

scores_rf_pca = cross_val_score(
    pipeline_rf_pca,
    X,
    y,
    cv=kfold,
    scoring='accuracy'
)

print(f"=== Random Forest with SVD (n_components={n_components}) ===")
print("Fold accuracies:", scores_rf_pca)
print("Mean accuracy:", scores_rf_pca.mean())
print("Std deviation:", scores_rf_pca.std())


=== Random Forest with SVD (n_components=5) ===
Fold accuracies: [1. 1. 0.]
Mean accuracy: 0.6666666666666666
Std deviation: 0.4714045207910317


Gradient Boosting with SVD

In [28]:
gb_pca = GradientBoostingClassifier(
    random_state=42
)

pipeline_gb_pca = Pipeline([
    ("svd", TruncatedSVD(n_components=n_components, random_state=42)),
    ("clf", gb_pca)
])

scores_gb_pca = cross_val_score(
    pipeline_gb_pca,
    X_dense if not issparse(X) else X,  # SVD can take sparse directly
    y,
    cv=kfold,
    scoring='accuracy'
)

print(f"=== Gradient Boosting with SVD (n_components={n_components}) ===")
print("Fold accuracies:", scores_gb_pca)
print("Mean accuracy:", scores_gb_pca.mean())
print("Std deviation:", scores_gb_pca.std())


=== Gradient Boosting with SVD (n_components=5) ===
Fold accuracies: [ 0.  0. nan]
Mean accuracy: nan
Std deviation: nan


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
1 fits failed out of a total of 3.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last_

Summary

In [29]:
results = {
    "LogReg (no PCA)": (scores_lr.mean(), scores_lr.std()),
    "RandomForest (no PCA)": (scores_rf.mean(), scores_rf.std()),
    "GradBoost (no PCA)": (scores_gb.mean(), scores_gb.std()),
    f"LogReg + SVD (k={n_components})": (scores_lr_pca.mean(), scores_lr_pca.std()),
    f"RandomForest + SVD (k={n_components})": (scores_rf_pca.mean(), scores_rf_pca.std()),
    f"GradBoost + SVD (k={n_components})": (scores_gb_pca.mean(), scores_gb_pca.std())
}

print("=== Model Comparison (Mean Accuracy ± Std) ===")
for name, (mean_acc, std_acc) in results.items():
    print(f"{name:30s}: {mean_acc:.3f} ± {std_acc:.3f}")

=== Model Comparison (Mean Accuracy ± Std) ===
LogReg (no PCA)               : nan ± nan
RandomForest (no PCA)         : 0.667 ± 0.471
GradBoost (no PCA)            : nan ± nan
LogReg + SVD (k=5)            : nan ± nan
RandomForest + SVD (k=5)      : 0.667 ± 0.471
GradBoost + SVD (k=5)         : nan ± nan
